# Vision Transformer (ViT) from scratch

**Paper:** An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale
**Difficulty:** medium · **Estimated time:** 12 hours

## Why this exercise
This is the big-pipeline warmup: ViT forces you to build every part of a real transformer (patch embed, attention, MLP, residual block, full encoder) plus the surrounding training + evaluation pipeline.

## Learning objective
Implement the full ViT architecture and the end-to-end training/evaluation pipeline so a single python assignment.py call trains it and reports a validation accuracy.

## Context
ViT showed that a transformer with no convolutions and no recurrence can match or beat CNNs on image classification when applied to sequences of image patches. You will reproduce the architecture and the pipeline around it at a tiny scale that runs on CPU.

## Prerequisites
- Tensor shapes and broadcasting
- nn.Module, forward(), and parameter registration
- Cross-entropy loss and softmax
- Adam/AdamW and basic training loop structure

## Concept ladder
1. Split an image into non-overlapping patches and project each patch into a vector
2. Prepend a learnable [CLS] token and add positional embeddings
3. Run the token sequence through stacked transformer blocks (multi-head self-attention + MLP + residuals + LayerNorm)
4. Classify from the [CLS] output and train with cross-entropy
5. Wrap data, model, optimizer, training and evaluation in one run_pipeline() that returns metrics

## Tasks
- Task 1: Implement PatchEmbedding, MultiHeadSelfAttention, MLPBlock, and TransformerBlock.
- Task 2: Implement the full ViT class (patch embed + [CLS] + pos embed + transformer stack + classifier head).
- Task 3: Implement get_dataloaders, train_one_epoch, and evaluate.
- Task 4: Implement run_pipeline that ties everything together and returns {train_loss, val_accuracy}.

## Milestones
1. **Forward pass returns logits**
   - Goal: A random batch flows through the full ViT and produces (B, num_classes) logits with no NaNs.
   - Check: `test_vit_forward_shape passes.`
2. **The architecture can learn**
   - Goal: Cross-entropy loss on a single fixed batch drops sharply over a few steps of AdamW.
   - Check: `test_vit_overfits_one_batch passes.`
3. **End-to-end pipeline runs**
   - Goal: run_pipeline() builds data, trains the model, evaluates, and returns metrics.
   - Check: `test_pipeline_runs_and_reports_metrics passes.`

## Setup
CPU is enough. No external dataset download - the tests use a tiny synthetic image dataset you build inside get_dataloaders. Use one file: assignment.py.

## How to work
- Implement the TODOs in the code cell below.
- Run the cell; the included tests identify the first failing requirement.
- Keep the implementation in one file-equivalent cell unless the exercise says otherwise.
- Finish only when the end-to-end pipeline and all checks pass.

In [ ]:
# Colab setup — PyTorch is normally preinstalled.
try:
    import torch
except ImportError:
    %pip install -q torch

import torch
print('PyTorch:', torch.__version__)
print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
# Project: Vision Transformer (ViT) from scratch
# Paper: An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale
# Goal: implement the full ViT architecture AND the end-to-end training pipeline
# Compute: CPU is enough at the tiny config used by the tests
# Rule: keep this as ONE file. Do not create vit.py, train.py, tests/, or extra modules.
# Run when done: python assignment.py

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


# =========================
# STUDENT CODE STARTS HERE
# =========================


class PatchEmbedding(nn.Module):
    def __init__(self, image_size, patch_size, in_channels, embed_dim):
        super().__init__()
        raise AssertionError("TODO: implement PatchEmbedding.__init__")

    def forward(self, x):
        raise AssertionError("TODO: implement PatchEmbedding.forward")


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        raise AssertionError("TODO: implement MultiHeadSelfAttention.__init__")

    def forward(self, x):
        raise AssertionError("TODO: implement MultiHeadSelfAttention.forward")


class MLPBlock(nn.Module):
    def __init__(self, embed_dim, mlp_ratio):
        super().__init__()
        raise AssertionError("TODO: implement MLPBlock.__init__")

    def forward(self, x):
        raise AssertionError("TODO: implement MLPBlock.forward")


class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio):
        super().__init__()
        raise AssertionError("TODO: implement TransformerBlock.__init__")

    def forward(self, x):
        raise AssertionError("TODO: implement TransformerBlock.forward")


class ViT(nn.Module):
    def __init__(self, image_size, patch_size, in_channels, num_classes, embed_dim, depth, num_heads, mlp_ratio):
        super().__init__()
        raise AssertionError("TODO: implement ViT.__init__")

    def forward(self, x):
        raise AssertionError("TODO: implement ViT.forward")


def get_dataloaders(num_train, num_val, image_size, in_channels, num_classes, batch_size):
    raise AssertionError("TODO: implement get_dataloaders")


def train_one_epoch(model, loader, optimizer, device):
    raise AssertionError("TODO: implement train_one_epoch")


def evaluate(model, loader, device):
    raise AssertionError("TODO: implement evaluate")


def run_pipeline():
    raise AssertionError("TODO: implement run_pipeline that wires data, model, optimizer, training and evaluation, and returns a dict with train_loss and val_accuracy")


# =========================
# STUDENT CODE ENDS HERE
# TESTS START BELOW
# =========================


TINY_CONFIG = dict(
    image_size=8,
    patch_size=4,
    in_channels=3,
    num_classes=4,
    embed_dim=32,
    depth=2,
    num_heads=4,
    mlp_ratio=2,
)


def test_vit_forward_shape():
    model = ViT(**TINY_CONFIG)
    x = torch.randn(2, TINY_CONFIG["in_channels"], TINY_CONFIG["image_size"], TINY_CONFIG["image_size"])
    logits = model(x)
    assert logits.shape == (2, TINY_CONFIG["num_classes"]), f"logits shape was {logits.shape}"
    assert torch.isfinite(logits).all(), "logits contain NaN/inf"


def test_vit_overfits_one_batch():
    torch.manual_seed(0)
    model = ViT(**TINY_CONFIG)
    x = torch.randn(8, TINY_CONFIG["in_channels"], TINY_CONFIG["image_size"], TINY_CONFIG["image_size"])
    y = torch.randint(0, TINY_CONFIG["num_classes"], (8,))
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
    initial_loss = F.cross_entropy(model(x), y).item()
    for _ in range(80):
        optimizer.zero_grad()
        loss = F.cross_entropy(model(x), y)
        loss.backward()
        optimizer.step()
    final_loss = F.cross_entropy(model(x), y).item()
    assert final_loss < 0.5 * initial_loss, (
        f"final loss {final_loss:.4f} should be much lower than initial {initial_loss:.4f}"
    )


def test_pipeline_runs_and_reports_metrics():
    torch.manual_seed(0)
    result = run_pipeline()
    assert isinstance(result, dict), "run_pipeline must return a dict"
    assert "val_accuracy" in result, "result must contain val_accuracy"
    assert "train_loss" in result, "result must contain train_loss"
    val_acc = float(result["val_accuracy"])
    assert 0.0 <= val_acc <= 1.0, f"val_accuracy out of range: {val_acc}"


def run_all_tests():
    print("Running ViT pipeline checks...")
    test_vit_forward_shape()
    print("1/3 ViT forward shape passed")
    test_vit_overfits_one_batch()
    print("2/3 model can overfit a single batch")
    test_pipeline_runs_and_reports_metrics()
    print("3/3 end-to-end pipeline returns metrics")
    print("All checks passed. You can mark this assignment done.")


if __name__ == "__main__":
    run_all_tests()
